# Company Risk Benchmarking

## Business question

Which companies show the highest complaint concentration and risk burden, and where should operational reviews start?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from cx_complaints.data.schema import normalize_complaints
from cx_complaints.features.text_features import add_text_features
from cx_complaints.modelling.risk import score_complaint_risk
from cx_complaints.preprocessing.text import add_clean_text_column

plt.style.use("ggplot")

## 1. Load and prepare complaint data

In [ ]:
raw_path = Path("data/raw/cfpb_sample.csv")
if not raw_path.exists():
    raise FileNotFoundError(
        "Expected data/raw/cfpb_sample.csv. Run the fetch command first."
    )

raw = pd.read_csv(raw_path)
complaints = normalize_complaints(raw)
complaints = add_clean_text_column(complaints)
complaints = add_text_features(complaints)
complaints = score_complaint_risk(complaints)
complaints.shape

## 2. Company benchmark table

In [ ]:
company_summary = (
    complaints.assign(company_clean=complaints["company"].fillna("Unknown").astype(str).str.strip())
    .groupby("company_clean")
    .agg(
        complaints=("complaint_id", "size"),
        avg_risk_score=("risk_score", "mean"),
        high_risk_share_pct=("risk_level", lambda s: (s == "high").mean() * 100),
        medium_or_high_share_pct=("risk_level", lambda s: s.isin(["medium", "high"]).mean() * 100),
        untimely_share_pct=("timely", lambda s: (s.fillna("").str.lower() != "yes").mean() * 100),
    )
    .reset_index()
    .sort_values(["complaints", "avg_risk_score"], ascending=[False, False])
)

company_summary["avg_risk_score"] = company_summary["avg_risk_score"].round(2)
for column in ["high_risk_share_pct", "medium_or_high_share_pct", "untimely_share_pct"]:
    company_summary[column] = company_summary[column].round(1)

company_summary.head(20)

## 3. Minimum-volume filter for stable comparisons

In [ ]:
min_volume = 20
company_stable = company_summary[company_summary["complaints"] >= min_volume].copy()
company_stable = company_stable.sort_values("high_risk_share_pct", ascending=False)

print(f"Companies with >= {min_volume} complaints: {len(company_stable)}")
company_stable.head(15)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_volume = company_summary.head(10).iloc[::-1]
axes[0].barh(top_volume["company_clean"], top_volume["complaints"], color="#2F6690")
axes[0].set_title("Top Companies by Complaint Volume")
axes[0].set_xlabel("Complaint count")

top_risk = company_stable.head(10).iloc[::-1]
axes[1].barh(top_risk["company_clean"], top_risk["high_risk_share_pct"], color="#B22222")
axes[1].set_title("Top Companies by High-Risk Share (Filtered)")
axes[1].set_xlabel("High-risk share (%)")

fig.tight_layout()

## 4. Product mix context by company

In [ ]:
focus_companies = company_summary.head(5)["company_clean"].tolist()
product_mix = (
    complaints.assign(
        company_clean=complaints["company"].fillna("Unknown").astype(str).str.strip(),
        product_clean=complaints["product"].fillna("Unknown").astype(str).str.strip(),
    )
    .query("company_clean in @focus_companies")
    .groupby(["company_clean", "product_clean"])
    .size()
    .rename("complaints")
    .reset_index()
)

product_mix_pivot = product_mix.pivot_table(
    index="company_clean",
    columns="product_clean",
    values="complaints",
    fill_value=0,
)

product_mix_share = product_mix_pivot.div(product_mix_pivot.sum(axis=1), axis=0).round(3)
product_mix_share

## 5. Interpretation guardrails

Use these safeguards before making company comparisons:

1. Apply minimum volume thresholds (`min_volume`) to avoid noisy percentages.
2. Compare within similar product mixes to reduce composition bias.
3. Treat this as complaint-intake signal, not market-share-adjusted performance.
4. Pair high-risk rates with narrative/topic evidence before action.

In [ ]:
# Optional export for stakeholder reporting
output_dir = Path("reports/notebook")
output_dir.mkdir(parents=True, exist_ok=True)

company_summary.to_csv(output_dir / "notebook_company_benchmark_full.csv", index=False)
company_stable.to_csv(output_dir / "notebook_company_benchmark_filtered.csv", index=False)
product_mix_share.to_csv(output_dir / "notebook_company_product_mix_share.csv")

output_dir